## Evaluation

In [3]:
from anatolian_sam.eval_utils import (
    plot_and_evaluate_3way,
    compute_emd_metrics,
    compute_si_sdr,
)
from pathlib import Path

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
import wandb

TRAINING_RUN_ID = input()

run = wandb.init(project="turkish-sam-audio", id=TRAINING_RUN_ID, resume="must")
eval_table = wandb.Table(
    columns=[
        "filename",
        "instrument",
        "domain",
        "emd_zero_shot",
        "emd_lora",
        "sisdr_zero_shot",
        "sisdr_lora",
        "gt_audio",
        "zs_audio",
        "lora_audio",
        "emd_plot",
    ]
)

epoch,34
global_step,2519
train/epoch_loss,0.30773
train/learning_rate,4e-05
train/step_loss,0.21269
val/epoch_loss,0.26849


In [19]:
# taken from vae.sample_rate and processor.sample_rate
# which is the sample rate used to train and inferenc the audio files
SAMPLE_RATE = 48000

In [20]:
import json
import os

import librosa
import torch
from tqdm import tqdm


def run_bulk_evaluation(
    val_metadata,
    gt_dir,
    separated_dir,
    eval_dir,
    domain_label,
    zs_prefix="zeroshot_",
    lora_prefix="lora_",
):
    """
    Evaluates a directory of test samples.
    """
    # load metadata
    metadata = []
    with open(val_metadata, "r") as f:
        for line in f:
            if line.strip():
                metadata.append(json.loads(line))

    gt_dir = Path(gt_dir)
    separated_dir = Path(separated_dir)

    # Create an output directory for the plots so they don't flood your notebook
    plot_dir = Path(eval_dir) / "plots" / domain_label
    plot_dir.mkdir(parents=True, exist_ok=True)

    results = []

    for stem in tqdm(metadata, desc=f"Evaluating {domain_label}"):
        filename = os.path.split(stem["target_path"])[-1]
        instrument = stem["prompt"]

        gt_audio_path = gt_dir / filename
        zs_audio_path = separated_dir / f"{zs_prefix}{filename}.wav"
        lora_audio_path = separated_dir / f"{lora_prefix}{filename}.wav"

        try:
            # Load audio
            y_gt, sr = librosa.load(gt_audio_path, sr=SAMPLE_RATE)
            y_zs, _ = librosa.load(zs_audio_path, sr=sr)
            y_lora, _ = librosa.load(lora_audio_path, sr=sr)

            # Find the shortest array length and trim all to that length to avoid errors in metric calculations
            min_length = min(len(y_gt), len(y_zs), len(y_lora))
            y_gt = y_gt[:min_length]
            y_zs = y_zs[:min_length]
            y_lora = y_lora[:min_length]

            # Compute metrics
            emd_zs, emd_lora, cents_tuple = compute_emd_metrics(y_gt, y_zs, y_lora, sr)
            sisdr_zs, sisdr_lora = compute_si_sdr(
                torch.from_numpy(y_gt),
                torch.from_numpy(y_zs),
                torch.from_numpy(y_lora),
                device="cpu",
            )

            # Store results as a dict to build the DataFrame efficiently later
            results.append(
                {
                    "filename": filename,
                    "instrument": instrument,
                    "domain": domain_label,
                    "emd_zero_shot": emd_zs,
                    "emd_lora": emd_lora,
                    "sisdr_zero_shot": sisdr_zs,
                    "sisdr_lora": sisdr_lora,
                }
            )

            # Optional: Save the plot to disk instead of rendering it inline
            cents_gt, cents_zs, cents_lora = cents_tuple
            plot_and_evaluate_3way(
                cents_gt, cents_zs, cents_lora, filename, plot_dir, save=True
            )

            # store results in a wandb table for later analysis
            eval_table.add_data(
                filename,
                instrument,
                domain_label,
                emd_zs,
                emd_lora,
                sisdr_zs,
                sisdr_lora,
                wandb.Audio(gt_audio_path, sample_rate=SAMPLE_RATE),
                wandb.Audio(zs_audio_path, sample_rate=SAMPLE_RATE),
                wandb.Audio(lora_audio_path, sample_rate=SAMPLE_RATE),
                wandb.Image(str(plot_dir / f"{filename}.png")),
            )

        except Exception as e:
            print(f"Error processing {filename}: {e}")
            raise e

    # Directly convert the list of dicts to a DataFrame
    return pd.DataFrame(results)

In [21]:
import pandas as pd


df_turkish = run_bulk_evaluation(
    val_metadata="../data/val_metadata_tr.jsonl",
    gt_dir="/teamspace/studios/turkish-music/anatolian-SAM/data/mixed",
    # gt_dir="../data/mixed",
    separated_dir="../evaluations/audio_results",
    eval_dir="../evaluations",
    domain_label="in_domain_turkish",
)

df_western = run_bulk_evaluation(
    val_metadata="../data/val_metadata_ww.jsonl",
    gt_dir="/teamspace/studios/turkish-music/anatolian-SAM/data/mixed/ww",
    # gt_dir="../data/mixed/ww",
    separated_dir="../evaluations/audio_results_ww",
    eval_dir="../evaluations",
    domain_label="out_of_domain_western",
)

df_final = pd.concat([df_turkish, df_western], ignore_index=True)

Evaluating in_domain_turkish:   0%|          | 0/144 [00:00<?, ?it/s]

Evaluating out_of_domain_western: 100%|██████████| 144/144 [12:53<00:00,  5.37s/it]


In [22]:
df_final.to_csv("../evaluations/evaluation_results.csv", index=False)

In [26]:
# Calculate the mean for BOTH EMD (Pitch) and SI-SDR (Cleanliness) across domains
matrix_summary = df_final.groupby("domain")[
    ["emd_zero_shot", "emd_lora", "sisdr_zero_shot", "sisdr_lora"]
].mean()

# Optional: Round the numbers to make the printed table look clean for your abstract
matrix_summary = matrix_summary.round(2)

print("\n--- Final 2x2 Experimental Matrix (EMD & SI-SDR) ---")
print(matrix_summary)


--- Final 2x2 Experimental Matrix (EMD & SI-SDR) ---
                       emd_zero_shot  emd_lora  sisdr_zero_shot  sisdr_lora
domain                                                                     
in_domain_turkish             107.83    762.96             1.82      -12.59
out_of_domain_western         524.25    587.08            -8.90      -10.90


In [28]:
# ==========================================
# W&B LOGGING PHASE
# ==========================================

# 1. Flatten the dataframe so 'domain' becomes a standard column
flat_matrix = matrix_summary.reset_index()

# 2. Convert the Pandas DataFrame directly into a W&B Table
eval_matrix_table = wandb.Table(dataframe=flat_matrix)

# 3. Log the table to your resumed training run
run.log(
    {
        "Metrics and Qualitative Data": eval_table,
        "Quantitative Final Matrix": eval_matrix_table,
    }
)

# 4. (Optional but highly recommended) Push these exact numbers to the run summary.
# This allows you to sort and filter your runs on the main W&B project dashboard!
for index, row in flat_matrix.iterrows():
    domain_name = row["domain"].replace(" ", "_").replace("-", "_").lower()

    # Example: run.summary["eval/in_domain_turkish/emd_lora"] = 120.45
    run.summary[f"eval/{domain_name}/emd_zero_shot"] = row["emd_zero_shot"]
    run.summary[f"eval/{domain_name}/emd_lora"] = row["emd_lora"]
    run.summary[f"eval/{domain_name}/sisdr_zero_shot"] = row["sisdr_zero_shot"]
    run.summary[f"eval/{domain_name}/sisdr_lora"] = row["sisdr_lora"]

print("Matrix successfully logged to W&B!")
run.finish()

Matrix successfully logged to W&B!


epoch,34
eval/in_domain_turkish/emd_lora,762.96
eval/in_domain_turkish/emd_zero_shot,107.83
eval/in_domain_turkish/sisdr_lora,-12.59
eval/in_domain_turkish/sisdr_zero_shot,1.82
eval/out_of_domain_western/emd_lora,587.08
eval/out_of_domain_western/emd_zero_shot,524.25
eval/out_of_domain_western/sisdr_lora,-10.9
eval/out_of_domain_western/sisdr_zero_shot,-8.9
global_step,2519
+4,...
